# Chess Coach Openings RAG

This notebook builds a retrieval augmented generation pipeline for chess opening coaching.

What it does:
- Searches Hugging Face for chess opening datasets and prefers `Lichess/chess-openings` when available.
- Uses opening-level chunks, which fit this dataset because each row is already one ECO/opening/variation record.
- Builds a local TF-IDF vector index instead of calling an embedding API.
- Retrieves with local vector search, BM25, and chess-specific keyword search.
- Merges retrieval results with reciprocal rank fusion.
- Answers through free OpenRouter chat models only, with a fallback list.
- Keeps user and assistant messages persistent in the active notebook session.

References checked on May 15, 2026:
- Hugging Face dataset: https://huggingface.co/datasets/Lichess/chess-openings
- OpenRouter free models: https://openrouter.ai/collections/free-models
- OpenRouter models API: https://openrouter.ai/docs/api/api-reference/models/get-models
- scikit-learn TF-IDF vectors: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

## Environment

A local `.venv` has been created at the project root with the current system `python3`. In VS Code or Jupyter, select the kernel from this environment after installing `ipykernel`.

If you need to register the kernel manually from a terminal:

```bash
source .venv/bin/activate
python -m pip install ipykernel
python -m ipykernel install --prefix .venv --name chess-coach-openings-rag --display-name "Chess Coach Openings RAG"
```

In [1]:
%pip install -q datasets huggingface_hub pandas numpy scikit-learn rank-bm25 python-dotenv requests ipywidgets tqdm joblib


Note: you may need to restart the kernel to use updated packages.


### Imports

In [2]:
from __future__ import annotations

import json
import math
import os
import re
import time
from html import escape
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import requests
from datasets import Dataset, DatasetDict, load_dataset
from dotenv import load_dotenv
from huggingface_hub import HfApi
from IPython.display import HTML, display
from rank_bm25 import BM25Okapi
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm


### Configuration

In [3]:
load_dotenv(".env")

HF_API_KEY = os.getenv("HF_API_KEY") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

CACHE_DIR = Path(".rag_cache")
LOCAL_VECTOR_INDEX_NAME = "tfidf_word_1_3_char_3_6"
TFIDF_WORD_NGRAM_RANGE = (1, 3)
TFIDF_CHAR_NGRAM_RANGE = (3, 6)
RRF_K = 60
RANDOM_SEED = 42


### Validate Configuration

In [4]:
missing = [
    name
    for name, value in {
        "HF_API_KEY": HF_API_KEY,
        "OPENROUTER_API_KEY": OPENROUTER_API_KEY,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(f"Missing required keys in .env: {', '.join(missing)}")

CACHE_DIR.mkdir(exist_ok=True)
np.random.seed(RANDOM_SEED)
print("Configuration loaded. Secrets are present but not displayed.")


Configuration loaded. Secrets are present but not displayed.


## Search Hugging Face For Chess Opening Data

The notebook searches Hugging Face first, then selects the best fit. `Lichess/chess-openings` is preferred because it is compact, CC0 licensed, and already has structured opening names, ECO codes, PGN, UCI, and EPD/FEN-like position data.

### Dataset Search Helpers

In [5]:
def _safe_get(obj: Any, name: str, default: Any = None) -> Any:
    return getattr(obj, name, default)


def search_chess_opening_datasets(query: str = "chess openings", limit: int = 20) -> pd.DataFrame:
    api = HfApi(token=HF_API_KEY)
    results = list(api.list_datasets(search=query, limit=limit))
    rows = []
    for item in results:
        dataset_id = _safe_get(item, "id", "")
        tags = _safe_get(item, "tags", []) or []
        tags_text = " ".join(tags).lower()
        downloads = _safe_get(item, "downloads", 0) or 0
        likes = _safe_get(item, "likes", 0) or 0
        score = math.log1p(downloads) + math.log1p(likes)
        if dataset_id == "Lichess/chess-openings":
            score += 100
        if "chess" in tags_text:
            score += 5
        if "open" in dataset_id.lower():
            score += 2
        rows.append(
            {
                "id": dataset_id,
                "downloads": downloads,
                "likes": likes,
                "tags": ", ".join(tags[:8]),
                "selection_score": score,
            }
        )
    return pd.DataFrame(rows).sort_values("selection_score", ascending=False).reset_index(drop=True)


### Search Candidate Datasets

In [6]:
dataset_candidates = search_chess_opening_datasets()
display(dataset_candidates.head(10))


,id,downloads,likes,tags,selection_score
0,Lichess/chess-openings,769,20,"license:cc0-1.0, size_categories:1K<n<10K, for...",116.690913
1,Lichess/antichess-chess-openings,18,1,"license:cc0-1.0, size_categories:n<1K, format:...",10.637586
2,coryvegan/chess-openings,16,0,"license:cc0-1.0, size_categories:1K<n<10K, for...",9.833213
3,Lichess/chess-openings-translations,55,1,"license:cc0-1.0, size_categories:1K<n<10K, for...",6.718499
4,coryvegan/Chess_openings_dataset,98,0,"task_categories:text-classification, task_cate...",6.595120
5,nelson2424/Chess_openings_dataset,92,0,"task_categories:text-classification, task_cate...",6.532599
6,dopamineaddict/dpo-chess-openings,30,0,"size_categories:10K<n<100K, format:json, modal...",5.433987
7,rakshit-nalayak/chess-openings-100k,1,0,"size_categories:10K<n<100K, format:json, modal...",2.693147


### Select Dataset

In [7]:
PREFERRED_DATASET_ID = "Lichess/chess-openings"

if PREFERRED_DATASET_ID in set(dataset_candidates["id"]):
    DATASET_ID = PREFERRED_DATASET_ID
elif len(dataset_candidates):
    DATASET_ID = dataset_candidates.iloc[0]["id"]
else:
    DATASET_ID = PREFERRED_DATASET_ID

print(f"Selected dataset: {DATASET_ID}")


Selected dataset: Lichess/chess-openings


### Dataset Loading Helpers

In [8]:
TEXT_COLUMNS = ["eco-volume", "eco", "name", "pgn", "uci", "epd"]
PARQUET_URL = "hf://datasets/Lichess/chess-openings/data/train-00000-of-00001.parquet"
IMAGE_COLUMNS = {"img", "image", "images"}
VIEWER_ROWS_URL = "https://datasets-server.huggingface.co/rows"


def load_lichess_openings_with_pandas() -> pd.DataFrame:
    """Fast path from the dataset card: read only text columns from the HF parquet file."""
    if DATASET_ID != "Lichess/chess-openings":
        raise ValueError("The direct parquet path is only known for Lichess/chess-openings.")
    return pd.read_parquet(
        PARQUET_URL,
        columns=TEXT_COLUMNS,
        storage_options={"token": HF_API_KEY} if HF_API_KEY else None,
    )


def drop_image_columns(row: dict[str, Any]) -> dict[str, Any]:
    return {key: value for key, value in row.items() if key.lower() not in IMAGE_COLUMNS}


def request_hf_rows_with_retries(params: dict[str, Any], headers: dict[str, str], retries: int = 5) -> dict[str, Any]:
    for attempt in range(retries):
        response = requests.get(VIEWER_ROWS_URL, headers=headers, params=params, timeout=60)
        if response.status_code not in {429, 500, 502, 503, 504}:
            response.raise_for_status()
            return response.json()
        wait_seconds = min(2 ** attempt, 16)
        print(f"HF rows request returned {response.status_code}; retrying in {wait_seconds}s...")
        time.sleep(wait_seconds)
    response.raise_for_status()
    return response.json()


def load_dataset_rows_via_viewer(
    dataset_id: str,
    config: str = "default",
    split: str = "train",
    batch_size: int = 50,
    max_rows: int | None = None,
) -> pd.DataFrame:
    """Load rows through the HF Dataset Viewer API without downloading image parquet payloads."""
    headers = {"Authorization": f"Bearer {HF_API_KEY}"} if HF_API_KEY else {}
    rows: list[dict[str, Any]] = []
    offset = 0
    total_rows = None
    progress = None

    while True:
        length = batch_size if max_rows is None else min(batch_size, max_rows - len(rows))
        if length <= 0:
            break

        payload = request_hf_rows_with_retries(
            params={
                "dataset": dataset_id,
                "config": config,
                "split": split,
                "offset": offset,
                "length": length,
            },
            headers=headers,
        )
        batch = [drop_image_columns(item.get("row", item)) for item in payload.get("rows", [])]

        if total_rows is None:
            total_rows = payload.get("num_rows_total")
            progress_total = min(total_rows, max_rows) if total_rows and max_rows else total_rows
            progress = tqdm(total=progress_total, desc="Requesting HF rows")

        if not batch:
            break

        rows.extend(batch)
        offset += len(batch)
        if progress:
            progress.update(len(batch))
        if total_rows and offset >= total_rows:
            break

    if progress:
        progress.close()
    return pd.DataFrame(rows)


def load_dataset_rows_streaming(dataset_id: str, split: str = "train", max_rows: int | None = None) -> pd.DataFrame:
    """Fallback loader. Streaming avoids a full local download, then drops image fields per row."""
    iterable = load_dataset(dataset_id, split=split, streaming=True, token=HF_API_KEY)
    rows = []
    for idx, row in enumerate(tqdm(iterable, desc="Streaming HF rows")):
        if max_rows is not None and idx >= max_rows:
            break
        rows.append(drop_image_columns(row))
    return pd.DataFrame(rows)


### Load Dataset

In [9]:
split_name = "train"
try:
    df = load_lichess_openings_with_pandas()
except Exception as exc:
    print(f"Direct parquet column load failed: {exc}")
    print("Falling back to Hugging Face Dataset Viewer API.")
    try:
        df = load_dataset_rows_via_viewer(DATASET_ID, config="default", split=split_name, batch_size=50)
    except Exception as viewer_exc:
        print(f"Dataset Viewer API load failed: {viewer_exc}")
        print("Falling back to Hugging Face streaming mode.")
        df = load_dataset_rows_streaming(DATASET_ID, split=split_name)

print(f"Loaded {len(df):,} text rows from split '{split_name}' without keeping image columns.")
display(df.head())


Loaded 3,627 text rows from split 'train' without keeping image columns.


,eco-volume,eco,name,pgn,uci,epd
0,A,A00,Amar Opening,1. Nh3,g1h3,rnbqkbnr/pppppppp/8/8/8/7N/PPPPPPPP/RNBQKB1R b...
1,A,A00,Amar Opening: Paris Gambit,1. Nh3 d5 2. g3 e5 3. f4,g1h3 d7d5 g2g3 e7e5 f2f4,rnbqkbnr/ppp2ppp/8/3pp3/5P2/6PN/PPPPP2P/RNBQKB...
2,A,A00,"Amar Opening: Paris Gambit, Gent Gambit",1. Nh3 d5 2. g3 e5 3. f4 Bxh3 4. Bxh3 exf4 5. ...,g1h3 d7d5 g2g3 e7e5 f2f4 c8h3 f1h3 e5f4 e1g1 f...,rn1qkbnr/ppp2ppp/8/3p4/8/6PB/PPPPP3/RNBQ1RK1 b...
3,A,A00,Amsterdam Attack,1. e3 e5 2. c4 d6 3. Nc3 Nc6 4. b3 Nf6,e2e3 e7e5 c2c4 d7d6 b1c3 b8c6 b2b3 g8f6,r1bqkb1r/ppp2ppp/2np1n2/4p3/2P5/1PN1P3/P2P1PPP...
4,A,A00,Anderssen's Opening,1. a3,a2a3,rnbqkbnr/pppppppp/8/8/8/P7/1PPPPPPP/RNBQKBNR b...


## Chunking Strategy

For chess openings, the cleanest chunk is usually one opening variation per row. A single Lichess row contains the opening name, ECO code, move order, UCI moves, and position. Splitting inside that record would separate the name from the line and hurt retrieval. The helper below still has a fallback text splitter for larger datasets with verbose move explanations.

### Chunking Helpers

In [10]:
def first_present(row: pd.Series, names: list[str], default: str = "") -> str:
    for name in names:
        if name in row and pd.notna(row[name]) and str(row[name]).strip():
            return str(row[name]).strip()
    return default


def row_to_document(row: pd.Series, row_id: int) -> dict[str, Any]:
    opening_name = first_present(row, ["name", "Opening_type", "opening", "title"])
    eco = first_present(row, ["eco", "ECO", "eco_code"])
    eco_volume = first_present(row, ["eco-volume", "eco_volume", "volume"])
    pgn = first_present(row, ["pgn", "PGN", "moves", "line"])
    uci = first_present(row, ["uci", "UCI"])
    epd = first_present(row, ["epd", "fen", "FEN"])
    context = first_present(row, ["Context", "context", "description", "text"])

    parts = []
    if opening_name:
        parts.append(f"Opening: {opening_name}")
    if eco:
        parts.append(f"ECO: {eco}")
    if eco_volume:
        parts.append(f"ECO volume: {eco_volume}")
    if pgn:
        parts.append(f"PGN move order: {pgn}")
    if uci:
        parts.append(f"UCI move order: {uci}")
    if epd:
        parts.append(f"Position EPD/FEN: {epd}")
    if context:
        parts.append(f"Dataset context: {context}")

    if not parts:
        parts = [json.dumps(row.dropna().to_dict(), ensure_ascii=True)]

    return {
        "doc_id": f"{DATASET_ID}:{row_id}",
        "opening_name": opening_name,
        "eco": eco,
        "pgn": pgn,
        "uci": uci,
        "epd": epd,
        "text": "\n".join(parts),
        "source_dataset": DATASET_ID,
    }


def split_long_text(text: str, max_chars: int = 1400, overlap: int = 180) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        boundary = max(text.rfind("\n", start, end), text.rfind(". ", start, end))
        if boundary > start + max_chars // 2:
            end = boundary + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return [chunk for chunk in chunks if chunk]


### Build Retrieval Documents

In [11]:
documents = []
for row_id, row in df.iterrows():
    base_doc = row_to_document(row, row_id)
    for chunk_id, chunk in enumerate(split_long_text(base_doc["text"])):
        doc = dict(base_doc)
        doc["chunk_id"] = chunk_id
        doc["text"] = chunk
        documents.append(doc)

docs_df = pd.DataFrame(documents)
print(f"Prepared {len(docs_df):,} retrieval chunks.")
display(docs_df.head())


Prepared 3,627 retrieval chunks.


,doc_id,opening_name,eco,pgn,uci,epd,text,source_dataset,chunk_id
0,Lichess/chess-openings:0,Amar Opening,A00,1. Nh3,g1h3,rnbqkbnr/pppppppp/8/8/8/7N/PPPPPPPP/RNBQKB1R b...,Opening: Amar Opening\nECO: A00\nECO volume: A...,Lichess/chess-openings,0
1,Lichess/chess-openings:1,Amar Opening: Paris Gambit,A00,1. Nh3 d5 2. g3 e5 3. f4,g1h3 d7d5 g2g3 e7e5 f2f4,rnbqkbnr/ppp2ppp/8/3pp3/5P2/6PN/PPPPP2P/RNBQKB...,Opening: Amar Opening: Paris Gambit\nECO: A00\...,Lichess/chess-openings,0
2,Lichess/chess-openings:2,"Amar Opening: Paris Gambit, Gent Gambit",A00,1. Nh3 d5 2. g3 e5 3. f4 Bxh3 4. Bxh3 exf4 5. ...,g1h3 d7d5 g2g3 e7e5 f2f4 c8h3 f1h3 e5f4 e1g1 f...,rn1qkbnr/ppp2ppp/8/3p4/8/6PB/PPPPP3/RNBQ1RK1 b...,"Opening: Amar Opening: Paris Gambit, Gent Gamb...",Lichess/chess-openings,0
3,Lichess/chess-openings:3,Amsterdam Attack,A00,1. e3 e5 2. c4 d6 3. Nc3 Nc6 4. b3 Nf6,e2e3 e7e5 c2c4 d7d6 b1c3 b8c6 b2b3 g8f6,r1bqkb1r/ppp2ppp/2np1n2/4p3/2P5/1PN1P3/P2P1PPP...,Opening: Amsterdam Attack\nECO: A00\nECO volum...,Lichess/chess-openings,0
4,Lichess/chess-openings:4,Anderssen's Opening,A00,1. a3,a2a3,rnbqkbnr/pppppppp/8/8/8/P7/1PPPPPPP/RNBQKBNR b...,Opening: Anderssen's Opening\nECO: A00\nECO vo...,Lichess/chess-openings,0


## Build Retrieval Indexes

This notebook uses a local TF-IDF vector index instead of remote embeddings. For chess openings, that is a better default because the important evidence is often exact or near-exact text: ECO codes, opening names, SAN/PGN moves, UCI moves, and FEN/EPD fragments. The index is cached in `.rag_cache/` and rebuilds only when the dataset or vector settings change.


### Text Search Helpers

In [12]:
TOKEN_PATTERN = re.compile(r"[a-zA-Z0-9+#=O\-]+")


def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace("0-0", "O-O").replace("0-0-0", "O-O-O")
    return re.sub(r"\s+", " ", text).strip()


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(normalize_text(text))


### Build BM25 Index

In [13]:
docs_df["search_text"] = docs_df["text"].map(normalize_text)
tokenized_corpus = docs_df["search_text"].map(tokenize).tolist()
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 and keyword corpus ready.")


BM25 and keyword corpus ready.


### Vector Index Helpers

In [14]:
def safe_slug(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", value).strip("_")


def build_local_vector_index(texts: list[str]) -> dict[str, Any]:
    word_vectorizer = TfidfVectorizer(
        token_pattern=TOKEN_PATTERN.pattern,
        ngram_range=TFIDF_WORD_NGRAM_RANGE,
        lowercase=True,
        sublinear_tf=True,
        dtype=np.float32,
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=TFIDF_CHAR_NGRAM_RANGE,
        sublinear_tf=True,
        dtype=np.float32,
    )

    word_matrix = word_vectorizer.fit_transform(texts)
    char_matrix = char_vectorizer.fit_transform(texts)
    doc_vectors = normalize(hstack([word_matrix, char_matrix], format="csr"), copy=False)
    return {
        "word_vectorizer": word_vectorizer,
        "char_vectorizer": char_vectorizer,
        "doc_vectors": doc_vectors,
    }


### Build Or Load Vector Index

In [15]:
vector_cache_path = CACHE_DIR / (
    f"vectors_{safe_slug(DATASET_ID)}_{safe_slug(LOCAL_VECTOR_INDEX_NAME)}_"
    f"n{len(docs_df)}.joblib"
)

if vector_cache_path.exists():
    local_vector_index = joblib.load(vector_cache_path)
    print(f"Loaded cached local vector index from {vector_cache_path}")
else:
    local_vector_index = build_local_vector_index(docs_df["search_text"].tolist())
    joblib.dump(local_vector_index, vector_cache_path)
    print(f"Saved local vector index to {vector_cache_path}")

doc_vectors = local_vector_index["doc_vectors"]
print(doc_vectors.shape)


Loaded cached local vector index from .rag_cache/vectors_Lichess_chess-openings_tfidf_word_1_3_char_3_6_n3627.joblib
(3627, 114090)


## Reciprocal Rank Fusion Retrieval

### Retrieval Helpers

In [16]:
query_vector_cache: dict[str, Any] = {}


def vectorize_query(query: str):
    cache_key = normalize_text(query)
    if cache_key not in query_vector_cache:
        word_vector = local_vector_index["word_vectorizer"].transform([cache_key])
        char_vector = local_vector_index["char_vectorizer"].transform([cache_key])
        query_vector_cache[cache_key] = normalize(
            hstack([word_vector, char_vector], format="csr"),
            copy=False,
        )
    return query_vector_cache[cache_key]


def top_indices(scores: np.ndarray, top_n: int) -> list[int]:
    top_n = min(top_n, len(scores))
    if top_n <= 0:
        return []
    candidate_idx = np.argpartition(-scores, top_n - 1)[:top_n]
    return candidate_idx[np.argsort(-scores[candidate_idx])].tolist()


def vector_ranking(query: str, pool_size: int = 80) -> tuple[list[int], np.ndarray]:
    q = vectorize_query(query)
    scores = np.asarray((doc_vectors @ q.T).todense()).reshape(-1)
    return top_indices(scores, pool_size), scores


def bm25_ranking(query: str, pool_size: int = 80) -> tuple[list[int], np.ndarray]:
    scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=np.float32)
    return top_indices(scores, pool_size), scores


def keyword_scores(query: str) -> np.ndarray:
    q_norm = normalize_text(query)
    q_tokens = set(tokenize(query))
    scores = np.zeros(len(docs_df), dtype=np.float32)
    for i, row in docs_df.iterrows():
        text = row["search_text"]
        name = normalize_text(row.get("opening_name", ""))
        eco = normalize_text(row.get("eco", ""))
        pgn = normalize_text(row.get("pgn", ""))
        if q_norm and q_norm in name:
            scores[i] += 15
        if q_norm and q_norm in text:
            scores[i] += 5
        if eco and eco in q_norm:
            scores[i] += 10
        if pgn and pgn in q_norm:
            scores[i] += 8
        doc_tokens = set(tokenized_corpus[i])
        scores[i] += len(q_tokens & doc_tokens)
    return scores


def keyword_ranking(query: str, pool_size: int = 80) -> tuple[list[int], np.ndarray]:
    scores = keyword_scores(query)
    return top_indices(scores, pool_size), scores


def reciprocal_rank_fusion(
    rankings: dict[str, list[int]],
    weights: dict[str, float] | None = None,
    k: int = RRF_K,
) -> list[tuple[int, float]]:
    weights = weights or {method: 1.0 for method in rankings}
    fused: dict[int, float] = {}
    for method, ranked_indices in rankings.items():
        weight = weights.get(method, 1.0)
        for rank, doc_idx in enumerate(ranked_indices, start=1):
            fused[doc_idx] = fused.get(doc_idx, 0.0) + weight / (k + rank)
    return sorted(fused.items(), key=lambda item: item[1], reverse=True)


def retrieve(query: str, top_k: int = 8, pool_size: int = 80) -> pd.DataFrame:
    vec_rank, vec_scores = vector_ranking(query, pool_size=pool_size)
    bm_rank, bm_scores = bm25_ranking(query, pool_size=pool_size)
    kw_rank, kw_scores = keyword_ranking(query, pool_size=pool_size)

    rankings = {"vector": vec_rank, "bm25": bm_rank, "keyword": kw_rank}
    fused = reciprocal_rank_fusion(
        rankings,
        weights={"vector": 1.15, "bm25": 1.0, "keyword": 1.25},
    )[:top_k]

    rows = []
    for doc_idx, fused_score in fused:
        row = docs_df.iloc[doc_idx].to_dict()
        row.update(
            {
                "doc_index": doc_idx,
                "rrf_score": fused_score,
                "vector_score": float(vec_scores[doc_idx]),
                "bm25_score": float(bm_scores[doc_idx]),
                "keyword_score": float(kw_scores[doc_idx]),
                "vector_rank": vec_rank.index(doc_idx) + 1 if doc_idx in vec_rank else None,
                "bm25_rank": bm_rank.index(doc_idx) + 1 if doc_idx in bm_rank else None,
                "keyword_rank": kw_rank.index(doc_idx) + 1 if doc_idx in kw_rank else None,
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


### Retrieval Preview

In [17]:
retrieval_preview = retrieve("What should I know about the Sicilian Defense Najdorf?", top_k=5)
display(retrieval_preview[["opening_name", "eco", "pgn", "rrf_score", "vector_rank", "bm25_rank", "keyword_rank"]])


,opening_name,eco,pgn,rrf_score,vector_rank,bm25_rank,keyword_rank
0,Sicilian Defense: Najdorf Variation,B90,1. e4 c5 2. Nf3 d6 3. d4 cxd4 4. Nxd4 Nf6 5. N...,0.055473,1,2,1
1,"Sicilian Defense: Najdorf Variation, English A...",B90,1. e4 c5 2. Nf3 d6 3. d4 cxd4 4. Nxd4 Nf6 5. N...,0.052425,4,7,4
2,Sicilian Defense: Najdorf Variation,B94,1. e4 c5 2. Nf3 d6 3. d4 cxd4 4. Nxd4 Nf6 5. N...,0.051065,2,4,14
3,"Sicilian Defense: Najdorf Variation, Lipnitsky...",B90,1. e4 c5 2. Nf3 d6 3. d4 cxd4 4. Nxd4 Nf6 5. N...,0.050972,7,6,7
4,"Sicilian Defense: Najdorf Variation, Adams Attack",B90,1. e4 c5 2. Nf3 d6 3. d4 cxd4 4. Nxd4 Nf6 5. N...,0.050626,12,9,2


## Free OpenRouter Model Selection

The notebook asks the OpenRouter models API for the current model list, filters to free text-output models, and then uses a preference order for chess coaching. The fallback `openrouter/free` route is kept last so the notebook can still work when specific free models change.

### OpenRouter Model Helpers

In [18]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

PREFERRED_FREE_MODEL_IDS = [
    "openrouter/owl-alpha",
    "deepseek/deepseek-v4-flash:free",
    "google/gemma-4-31b-it:free",
    "openai/gpt-oss-20b:free",
    "qwen/qwen3-coder:free",
    "meta-llama/llama-3.3-70b-instruct:free",
    "openrouter/free",
]


def _price_is_zero(value: Any) -> bool:
    try:
        return float(value) == 0.0
    except (TypeError, ValueError):
        return False


def list_free_openrouter_models() -> list[dict[str, Any]]:
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    response = requests.get(
        f"{OPENROUTER_BASE_URL}/models",
        headers=headers,
        params={"output_modalities": "text"},
        timeout=30,
    )
    response.raise_for_status()
    models = response.json().get("data", [])
    free_models = []
    for model in models:
        model_id = model.get("id", "")
        name = model.get("name", "")
        architecture = model.get("architecture") or {}
        pricing = model.get("pricing") or {}
        output_modalities = architecture.get("output_modalities") or []
        is_text_output = not output_modalities or "text" in output_modalities
        is_free = (
            model_id.endswith(":free")
            or "free" in name.lower()
            or (
                _price_is_zero(pricing.get("prompt"))
                and _price_is_zero(pricing.get("completion"))
                and _price_is_zero(pricing.get("request", 0))
            )
        )
        if is_text_output and is_free:
            free_models.append(model)
    return free_models


### Select Generation Models

In [19]:
try:
    free_models = list_free_openrouter_models()
except Exception as exc:
    print(f"Could not fetch live OpenRouter models: {exc}")
    free_models = []

free_model_ids = {model.get("id") for model in free_models}
GENERATION_MODELS = [model_id for model_id in PREFERRED_FREE_MODEL_IDS if model_id in free_model_ids]

if "openrouter/free" not in GENERATION_MODELS:
    GENERATION_MODELS.append("openrouter/free")

print("Free OpenRouter models selected for fallback order:")
for model_id in GENERATION_MODELS:
    print("-", model_id)

free_models_preview = pd.DataFrame(
    [
        {
            "id": model.get("id"),
            "name": model.get("name"),
            "context_length": model.get("context_length"),
        }
        for model in free_models
    ]
)
display(free_models_preview.head(20))


Free OpenRouter models selected for fallback order:
- openrouter/owl-alpha
- deepseek/deepseek-v4-flash:free
- google/gemma-4-31b-it:free
- openai/gpt-oss-20b:free
- qwen/qwen3-coder:free
- meta-llama/llama-3.3-70b-instruct:free
- openrouter/free


,id,name,context_length
0,baidu/cobuddy:free,Baidu Qianfan: CoBuddy (free),131072
1,openrouter/owl-alpha,Owl Alpha,1048756
2,nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:...,NVIDIA: Nemotron 3 Nano Omni (free),256000
3,poolside/laguna-xs.2:free,Poolside: Laguna XS.2 (free),131072
4,poolside/laguna-m.1:free,Poolside: Laguna M.1 (free),131072
5,deepseek/deepseek-v4-flash:free,DeepSeek: DeepSeek V4 Flash (free),1048576
6,google/gemma-4-26b-a4b-it:free,Google: Gemma 4 26B A4B (free),262144
7,google/gemma-4-31b-it:free,Google: Gemma 4 31B (free),262144
8,arcee-ai/trinity-large-thinking:free,Arcee AI: Trinity Large Thinking (free),262144
9,google/lyria-3-pro-preview,Google: Lyria 3 Pro Preview,1048576


## Answer Generation

The next helper cell owns display formatting. The following core cell owns retrieval, model calling, session memory, and the `ask()` entry point.


### Answer Display Helpers


In [20]:
from html import escape
from IPython.display import HTML, display
from typing import Any


def plain_text_response(text: str) -> str:
    """Remove common Markdown artifacts when a free model ignores the plain-text instruction."""
    text = text.strip()
    text = re.sub(r"```(?:\w+)?\n?(.*?)```", r"\1", text, flags=re.S)
    text = re.sub(r"^\s*#{1,6}\s+", "", text, flags=re.M)
    text = re.sub(r"^\s*[-*+]\s+", "", text, flags=re.M)
    text = re.sub(r"^\s*>\s?", "", text, flags=re.M)
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)
    text = re.sub(r"__(.*?)__", r"\1", text)
    text = re.sub(r"(?<!\*)\*([^*\n]+)\*(?!\*)", r"\1", text)
    text = re.sub(r"(?<!_)_([^_\n]+)_(?!_)", r"\1", text)
    text = re.sub(r"`([^`]*)`", r"\1", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = re.sub(r"^\s*\|?\s*:?-{3,}:?\s*(\|\s*:?-{3,}:?\s*)+\|?\s*$", "", text, flags=re.M)
    text = re.sub(r"^\|(.+)\|$", lambda match: "  ".join(part.strip() for part in match.group(1).split("|")), text, flags=re.M)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def compact_source_value(value: Any, digits: int | None = None) -> str:
    if pd.isna(value):
        return "-"
    if digits is not None:
        return f"{float(value):.{digits}f}"
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def render_answer_panel(question: str, answer: str, model_used: str) -> HTML:
    return HTML(
        f"""
        <div style="border:1px solid #2f4f3f; border-radius:8px; overflow:hidden; margin:10px 0 14px; background:#101712; font-family:system-ui, -apple-system, Segoe UI, sans-serif;">
          <div style="display:flex; align-items:center; justify-content:space-between; gap:12px; padding:10px 14px; background:linear-gradient(90deg, #16251b, #203528); border-bottom:1px solid #2f4f3f;">
            <div style="color:#f4e8c1; font-weight:700; letter-spacing:0;">Chess Opening Coach</div>
            <div style="color:#b9c7b3; font-size:12px;">Model: {escape(model_used)}</div>
          </div>
          <div style="padding:12px 14px; border-bottom:1px solid #263b30; color:#d8e0d2;">
            <span style="color:#8fbf95; font-size:12px; font-weight:700; text-transform:uppercase;">Question</span><br>
            <span>{escape(question)}</span>
          </div>
          <pre style="white-space:pre-wrap; margin:0; padding:14px; color:#f4f1e8; background:#0f1411; font:14px/1.55 ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;">{escape(answer)}</pre>
        </div>
        """
    )


def render_sources_table(results: pd.DataFrame) -> HTML:
    rows = []
    for source_rank, (_, row) in enumerate(results.iterrows(), start=1):
        score = compact_source_value(row.get("rrf_score"), digits=4)
        opening = escape(compact_source_value(row.get("opening_name")))
        eco = escape(compact_source_value(row.get("eco")))
        pgn = escape(compact_source_value(row.get("pgn")))
        vector_rank = escape(compact_source_value(row.get("vector_rank")))
        bm25_rank = escape(compact_source_value(row.get("bm25_rank")))
        keyword_rank = escape(compact_source_value(row.get("keyword_rank")))
        rows.append(
            f"""
            <tr>
              <td class="source-rank">{source_rank}</td>
              <td><div class="source-opening">{opening}</div><div class="source-pgn">{pgn}</div></td>
              <td class="source-eco">{eco}</td>
              <td class="source-score">{score}</td>
              <td class="source-ranks">V {vector_rank}<br>B {bm25_rank}<br>K {keyword_rank}</td>
            </tr>
            """
        )
    return HTML(
        f"""
        <style>
          .rag-sources {{ border:1px solid #2f4f3f; border-radius:8px; overflow:hidden; margin:12px 0 4px; font-family:system-ui, -apple-system, Segoe UI, sans-serif; background:#111814; }}
          .rag-sources-title {{ display:flex; justify-content:space-between; gap:12px; align-items:center; padding:10px 14px; color:#f4e8c1; background:linear-gradient(90deg, #16251b, #203528); border-bottom:1px solid #2f4f3f; font-weight:700; }}
          .rag-sources-subtitle {{ color:#b9c7b3; font-size:12px; font-weight:500; }}
          .rag-sources table {{ width:100%; border-collapse:collapse; table-layout:fixed; }}
          .rag-sources th {{ text-align:left; padding:9px 10px; color:#b9c7b3; background:#0d120f; font-size:12px; text-transform:uppercase; border-bottom:1px solid #2f4f3f; }}
          .rag-sources td {{ padding:10px; color:#e8e2d0; border-bottom:1px solid #24382d; vertical-align:top; }}
          .rag-sources tr:nth-child(even) td {{ background:#151d18; }}
          .rag-sources tr:last-child td {{ border-bottom:0; }}
          .source-rank {{ width:42px; color:#f4e8c1; font-weight:700; text-align:center; }}
          .source-opening {{ font-weight:700; color:#f4f1e8; margin-bottom:4px; }}
          .source-pgn {{ color:#c9c2af; font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size:12px; white-space:normal; overflow-wrap:anywhere; }}
          .source-eco {{ width:72px; color:#8fbf95; font-weight:700; }}
          .source-score {{ width:88px; color:#f4e8c1; font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; }}
          .source-ranks {{ width:74px; color:#b9c7b3; font-family:ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size:12px; }}
        </style>
        <div class="rag-sources">
          <div class="rag-sources-title"><span>Retrieved Sources</span><span class="rag-sources-subtitle">fusion score and retrieval ranks</span></div>
          <table>
            <thead><tr><th style="width:42px; text-align:center;">#</th><th>Opening and line</th><th style="width:72px;">ECO</th><th style="width:88px;">Score</th><th style="width:74px;">Ranks</th></tr></thead>
            <tbody>{''.join(rows)}</tbody>
          </table>
        </div>
        """
    )


### Core Answer Logic

In [21]:
SYSTEM_PROMPT = """
You are a practical chess opening coach. Answer from the retrieved opening records first.
Return plain text only. Do not use Markdown, bullet markers, tables, code fences, headings,
bold text, italic text, or inline code formatting. Use short labeled lines like Opening,
Main line, White plan, Black plan, Risks, and Recommendation. Give concrete move orders,
ECO codes when available, plans for both sides, typical risks, and one or two beginner-friendly
recommendations. If the retrieved data is insufficient, say what is missing instead of inventing a line.
""".strip()

SESSION_MESSAGES: list[dict[str, str]] = []


def format_context(results: pd.DataFrame) -> str:
    blocks = []
    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        blocks.append(
            "\n".join(
                [
                    f"[Source {rank}]",
                    f"Opening: {row.get('opening_name', '')}",
                    f"ECO: {row.get('eco', '')}",
                    f"PGN: {row.get('pgn', '')}",
                    f"UCI: {row.get('uci', '')}",
                    f"EPD/FEN: {row.get('epd', '')}",
                    f"Text: {row.get('text', '')}",
                ]
            )
        )
    return "\n\n".join(blocks)


def call_openrouter_chat(messages: list[dict[str, str]], models: list[str] | None = None) -> tuple[str, str]:
    models = models or GENERATION_MODELS
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "Chess Coach Openings RAG",
    }
    last_error = None
    for model_id in models:
        payload = {
            "model": model_id,
            "messages": messages,
            "temperature": 0.35,
            "max_tokens": 900,
        }
        try:
            response = requests.post(
                f"{OPENROUTER_BASE_URL}/chat/completions",
                headers=headers,
                json=payload,
                timeout=90,
            )
            if response.status_code in {429, 500, 502, 503, 504}:
                last_error = RuntimeError(f"{model_id}: {response.status_code} {response.text[:300]}")
                continue
            response.raise_for_status()
            data = response.json()
            answer = data["choices"][0]["message"]["content"]
            return answer, model_id
        except Exception as exc:
            last_error = exc
            continue
    raise RuntimeError(f"All OpenRouter free model calls failed. Last error: {last_error}")


def ask(question: str, top_k: int = 8, show_sources: bool = True, verbose: bool = True) -> str:
    results = retrieve(question, top_k=top_k)
    context = format_context(results)
    recent_history = SESSION_MESSAGES[-8:]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *recent_history,
        {
            "role": "user",
            "content": (
                "Retrieved chess opening records:\n"
                f"{context}\n\n"
                f"Question: {question}"
            ),
        },
    ]
    answer, model_used = call_openrouter_chat(messages)
    answer = plain_text_response(answer)
    SESSION_MESSAGES.append({"role": "user", "content": question})
    SESSION_MESSAGES.append({"role": "assistant", "content": answer})

    if verbose:
        display(render_answer_panel(question, answer, model_used))
    if show_sources:
        display(render_sources_table(results))
    return answer


def clear_chat_history(verbose: bool = True) -> None:
    SESSION_MESSAGES.clear()
    if verbose:
        print("Session chat history cleared.")


In [22]:
# Example question. Edit and rerun.
_ = ask("tell me about the english opening", top_k=8)


#,Opening and line,ECO,Score,Ranks
1,English Opening: The Whale1. e4 e5 2. c4,C20,0.0557,V 1B 1K 1
2,Dresden Opening: The Goblin1. e4 e5 2. Nf3 Nc6 3. c4 Nf6 4. Nxe5,C44,0.0445,V 67B 4K 3
3,"English Opening: King's English Variation, Two Knights Variation1. c4 e5 2. Nc3 Nf6",A22,0.0416,V 5B 11K 67
4,"English Opening: King's English Variation, Reversed Closed Sicilian1. c4 e5 2. Nc3 Nc6",A25,0.0392,V 10B 12K 80
5,"English Opening: King's English Variation, Four Knights Variation, Fianchetto Line1. c4 e5 2. g3 Nf6 3. Bg2 d5 4. cxd5 Nxd5 5. Nf3 Nc6",A29,0.0373,V 25B 40K 31
6,"English Opening: King's English, Mazedonisch1. c4 e5 2. Nc3 Nf6 3. f4",A22,0.0371,V 23B 13K 71
7,"English Opening: King's English Variation, Two Knights Variation, Keres Variation1. c4 e5 2. Nc3 Nf6 3. g3 c6",A23,0.0368,V 16B 22K 72
8,"English Opening: King's English Variation, Two Knights Variation, Reversed Dragon1. c4 e5 2. Nc3 Nf6 3. g3 d5",A22,0.0367,V 17B 23K 68


## Notebook Chatbot UI

Run the next cells for a session-only chatbot inside the notebook. Messages stay in `SESSION_MESSAGES` until you restart the kernel, click Clear, or call `clear_chat_history()`.

### Chat Display Helpers


In [23]:
import ipywidgets as widgets
from html import escape
from IPython.display import display
from typing import Any


CHAT_SHELL_STYLE = """
<div style="border:1px solid #314f3f; border-radius:8px; overflow:hidden; background:#0f1411; font-family:system-ui, -apple-system, Segoe UI, sans-serif;">
  <div style="display:flex; align-items:stretch; min-height:86px; background:linear-gradient(90deg, #122016, #1d3326); border-bottom:1px solid #314f3f;">
    <div style="width:86px; flex:0 0 86px; background-color:#d8c99b; background-image:linear-gradient(45deg,#35513f 25%,transparent 25%),linear-gradient(-45deg,#35513f 25%,transparent 25%),linear-gradient(45deg,transparent 75%,#35513f 75%),linear-gradient(-45deg,transparent 75%,#35513f 75%); background-size:34px 34px; background-position:0 0,0 17px,17px -17px,-17px 0;"></div>
    <div style="padding:16px 18px; display:flex; flex-direction:column; justify-content:center; gap:4px;">
      <div style="color:#f4e8c1; font-size:20px; font-weight:800; letter-spacing:0;">Chess Opening Coach</div>
      <div style="color:#b9c7b3; font-size:13px;">Session-only chat for opening plans, move orders, and risks</div>
    </div>
  </div>
</div>
"""


CHAT_FOOTER_STYLE = """
<div style="height:10px; border-left:1px solid #314f3f; border-right:1px solid #314f3f; border-bottom:1px solid #314f3f; border-radius:0 0 8px 8px; background:#0f1411;"></div>
"""


def chat_bubble(role: str, text: str) -> widgets.HTML:
    is_user = role == "user"
    align = "flex-end" if is_user else "flex-start"
    label = "You" if is_user else "Coach"
    background = "#214f39" if is_user else "#f4eedc"
    color = "#ffffff" if is_user else "#20231f"
    border = "1px solid #407a5b" if is_user else "1px solid #d8c99b"
    label_color = "#d8f0da" if is_user else "#47624d"
    return widgets.HTML(
        value=f"""
        <div style="display:flex; justify-content:{align}; margin:10px 0;">
          <div style="max-width:82%; white-space:pre-wrap; background:{background}; color:{color}; border:{border}; padding:10px 12px; border-radius:8px; line-height:1.48; box-shadow:0 2px 10px rgba(0,0,0,0.18); font-family:system-ui, -apple-system, Segoe UI, sans-serif;">
            <div style="font-size:12px; font-weight:800; color:{label_color}; margin-bottom:5px; text-transform:uppercase; letter-spacing:0;">{label}</div>
            <div>{escape(text)}</div>
          </div>
        </div>
        """
    )


### Chat Logic Helper

In [24]:
def launch_chatbot() -> None:
    question_box = widgets.Text(
        placeholder="Ask about Ruy Lopez plans, Sicilian traps, English move orders...",
        continuous_update=False,
        layout=widgets.Layout(width="100%"),
    )
    send_button = widgets.Button(description="Send", button_style="primary", icon="paper-plane", layout=widgets.Layout(width="110px"))
    clear_button = widgets.Button(description="Clear", icon="trash", layout=widgets.Layout(width="98px"))
    status = widgets.HTML(value="")
    messages_box = widgets.VBox(
        layout=widgets.Layout(
            width="100%",
            min_height="260px",
            max_height="520px",
            overflow_y="auto",
            padding="12px 14px",
            border="1px solid #314f3f",
        )
    )
    input_row = widgets.HBox([question_box, send_button], layout=widgets.Layout(width="100%", gap="8px", padding="10px 0 6px"))

    def append_message(role: str, text: str) -> None:
        messages_box.children = (*messages_box.children, chat_bubble(role, text))

    def send_message(_event: Any = None) -> None:
        question = question_box.value.strip()
        if not question or send_button.disabled:
            return
        question_box.value = ""
        append_message("user", question)
        send_button.disabled = True
        question_box.disabled = True
        status.value = "<span style='color:#b9c7b3; font-size:13px;'>Coach is analyzing candidate lines...</span>"
        try:
            answer = ask(question, top_k=8, show_sources=False, verbose=False)
            append_message("assistant", answer)
        except Exception as exc:
            append_message("assistant", f"Error: {exc}")
        finally:
            status.value = ""
            send_button.disabled = False
            question_box.disabled = False
            question_box.focus()

    def on_text_submit(change: dict[str, Any]) -> None:
        if change.get("new", "").strip():
            send_message()

    def on_clear(_button: widgets.Button) -> None:
        clear_chat_history(verbose=False)
        messages_box.children = ()
        status.value = ""
        question_box.value = ""

    question_box.observe(on_text_submit, names="value")
    send_button.on_click(send_message)
    clear_button.on_click(on_clear)
    controls = widgets.HBox([clear_button, status], layout=widgets.Layout(width="100%", align_items="center", gap="10px"))
    display(widgets.VBox([widgets.HTML(CHAT_SHELL_STYLE), messages_box, input_row, controls, widgets.HTML(CHAT_FOOTER_STYLE)], layout=widgets.Layout(width="100%")))


### Launch Chatbot

In [25]:
launch_chatbot()
